In [9]:
import pandas as pd

# load dataset 
df = pd.read_csv(r"C:\Users\kazmi\Downloads\project_file\AI Job Market Dataset.csv")

In [3]:
# Data Overview
df.info()
print("Missing Values:\n", df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10345 entries, 0 to 10344
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   job_id                10345 non-null  int64 
 1   job_title             10345 non-null  object
 2   company_size          10345 non-null  object
 3   company_industry      10345 non-null  object
 4   country               10345 non-null  object
 5   remote_type           10345 non-null  object
 6   experience_level      10345 non-null  object
 7   years_experience      10345 non-null  int64 
 8   education_level       10345 non-null  object
 9   skills_python         10345 non-null  int64 
 10  skills_sql            10345 non-null  int64 
 11  skills_ml             10345 non-null  int64 
 12  skills_deep_learning  10345 non-null  int64 
 13  skills_cloud          10345 non-null  int64 
 14  salary                10345 non-null  int64 
 15  job_posting_month     10345 non-null

In [4]:
# Remove duplicates
print("Duplicates_before", df.duplicated().sum())
df = df.drop_duplicates()

Duplicates_before 0


In [5]:
# Check Data types
print(df.dtypes)

job_id                   int64
job_title               object
company_size            object
company_industry        object
country                 object
remote_type             object
experience_level        object
years_experience         int64
education_level         object
skills_python            int64
skills_sql               int64
skills_ml                int64
skills_deep_learning     int64
skills_cloud             int64
salary                   int64
job_posting_month        int64
job_posting_year         int64
hiring_urgency          object
job_openings             int64
dtype: object


In [6]:
# Data type fix
categorical_cols = [
    "country", "remote_type", "experience_level",
    "company_size", "education_level"
]
for col in categorical_cols:
    df[col] = df[col].astype("category")

In [7]:
# Feature engineering
skill_cols = [
    "skills_python",
    "skills_sql",
    "skills_ml",
    "skills_deep_learning",
    "skills_cloud"
]

df[skill_cols] = df[skill_cols].fillna(0).astype(int)

df["total_skills"] = df[skill_cols].sum(axis=1)

df["years_experience"] = df["years_experience"].clip(lower=0)


df["experience_category"] = pd.cut(
    df["years_experience"],
    bins=[0,3,7,float("inf")],
    labels=["Entry", "Mid", "Senior"],
    include_lowest=True
    )

In [8]:
# Final Validation
print("Missing Values:\n", df.isnull().sum())
print("Duplicate Rows :", df.duplicated().sum())

Missing Values:
 job_id                  0
job_title               0
company_size            0
company_industry        0
country                 0
remote_type             0
experience_level        0
years_experience        0
education_level         0
skills_python           0
skills_sql              0
skills_ml               0
skills_deep_learning    0
skills_cloud            0
salary                  0
job_posting_month       0
job_posting_year        0
hiring_urgency          0
job_openings            0
total_skills            0
experience_category     0
dtype: int64
Duplicate Rows : 0


In [9]:
#Preview
print(df.head())

   job_id                  job_title company_size company_industry    country  \
0       1                AI Engineer      Startup           Retail     Canada   
1       2  Machine Learning Engineer          MNC       Technology  Australia   
2       3  Machine Learning Engineer          MNC       Technology    Germany   
3       4           Business Analyst      Startup       Healthcare    Germany   
4       5             Data Scientist          MNC       Healthcare    Germany   

  remote_type experience_level  years_experience education_level  \
0      Remote           Senior                 2          Master   
1      Hybrid              Mid                 0        Bachelor   
2      Onsite              Mid                14          Master   
3      Remote              Mid                 9          Master   
4      Hybrid              Mid                 5          Master   

   skills_python  ...  skills_ml  skills_deep_learning  skills_cloud  salary  \
0              0  ...   

In [13]:
#Save Cleaned Dataset
df.to_csv("Cleaned_job_dataset.csv", index=False)

## Q1: Salary Drivers

### 1A: Analysis by Experience Level

In [6]:
# Avearge salary By experience level
df.groupby("experience_level")["salary"].mean().sort_values()

experience_level
Entry      89095.872758
Mid       113592.018464
Senior    138289.091520
Name: salary, dtype: float64

#### Insight
Salary shows a clear upward trend across experience levels, increasing from approximately ~89K at entry level to ~138K at senior level (~49K difference). This highlights experience level as a major driver of compensation in the job market.

### 1B: Analysis by company size

In [5]:
# Avearge salary by company size
df.groupby("company_size")["salary"].mean().sort_values()

company_size
Medium        107915.975667
Enterprise    108508.403061
Startup       108922.905120
MNC           128333.940609
Name: salary, dtype: float64

#### Insight
Company size shows minimal impact on salary overall, with Medium, Enterprise, and Startup roles clustered around ~108K. However, MNCs offer significantly higher salaries (~128K), indicating a ~20K premium, likely driven by greater resources and global scale.

### 1C: Analysis by Education level 

In [4]:
# Avearge salary by education level
df.groupby("education_level")["salary"].mean().sort_values()

education_level
PhD         112962.324380
Bachelor    113314.773223
Master      114018.731652
Name: salary, dtype: float64

#### Insight
Education level is not a strong indicator of salary, with all categories ranging narrowly between ~112.9K and ~114K. This minimal variation suggests that experience levels and technical skills play a more significant role in determining compensation than formal education.

### 1D: Analysis by total skills

In [8]:
# Avearge salary by total skills
df.groupby("total_skills")["salary"].mean().sort_values()

total_skills
0     91174.141379
1    102254.460890
2    109404.316384
3    117000.661356
4    125038.051051
5    134490.973607
Name: salary, dtype: float64

#### Insight
Salary increases consistently with the number of technical skills, rising from ~91K (0 skills) to ~134K (5 skills) — an increase of approximately 43K (~47%). This indicates a strong and near-linear relationship, highlighting skill intensity as one of the most influential drivers of compensation.

### 1E: Analysis by remote type

In [2]:
# Avearge salary by remote type
df.groupby("remote_type")["salary"].mean().sort_values()

remote_type
Remote    113305.101623
Hybrid    113363.757018
Onsite    113649.938453
Name: salary, dtype: float64

#### Insight
Salary variation across remote, hybrid, and onsite roles is negligible, with all categories clustered around ~113K. This suggests that compensation is largely independent of work mode and primarily driven by role requirements.

### Final Insight
The analysis highlights how different factors relate to salary levels. Salary differences are most pronounced across experience levels and total technical skills, indicating a strong relationship. In contrast, employment type and education levels exhibit minimal impact. Company size has a limited overall effect, with noticeably higher salaries observed in MNCs.

## Q2: Experience vs Salary Progression

### 2A: Analysis by experience-years

In [20]:
#Salary trend by years of experience
df.groupby("years_experience")["salary"].mean().sort_values()

years_experience
10    111090.147989
13    111603.990950
14    112183.957204
6     112269.121457
9     112893.534610
4     113154.974398
1     113436.471616
2     113673.095035
5     113838.097784
12    114179.672464
8     114243.895476
0     114282.148305
3     114742.500747
7     114778.481645
11    115386.104135
Name: salary, dtype: float64

#### Insight
Average salaries vary within a narrow range of ~111K to ~115K across years of experience (≈4K difference), indicating weak progression. Lower experience groups (0–3 years ≈114K) often match or exceed salaries of higher experience groups (10–14 years ≈111K–112K), showing no consistent upward trend. This suggests that years of experience alone has limited influence on salary.

### 2B: Analysis by experience category

In [21]:
#Salary by experience category
df.groupby("experience_category", observed="True")["salary"].mean().sort_values()

experience_category
Mid       113019.009599
Senior    113323.051348
Entry     113855.620476
Name: salary, dtype: float64

#### Insight
There is no clear upward trend, and salaries differ by less than ~1K across experience categories, suggesting that experience categories derived from years of experience have negligible impact on compensation.

### Final Insight
Salary progression across both individual years of experience and derived experience categories appears inconsistent and non-linear. Despite higher experience, salary differences remain minimal (~4K across years and less than ~1K across categories), indicating that experience alone has limited influence on compensation compared to other factors such as technical skills and job defined experience levels.

## Q3: Experience level consistency

### 3A: Count Analysis

In [ ]:
# Alignment of experience categories with job-defined experience levels
pd.crosstab(df["experience_level"], df["experience_category"])

experience_category,Entry,Mid,Senior
experience_level,,,
Entry,938,968,1607
Mid,952,915,1545
Senior,879,970,1571


### 3B: Percentage Analysis

In [ ]:
# Percentage distribution of experience categories within each job-defined experience level
pd.crosstab(
    df["experience_level"],
    df["experience_category"],
    normalize= "index"
).multiply(100).round(2)

experience_category,Entry,Mid,Senior
experience_level,,,
Entry,26.7,27.55,45.74
Mid,27.9,26.82,45.28
Senior,25.7,28.36,45.94


### 3C: Match vs Mismatch Analysis

In [16]:
# Compute match and mismatch records between job-defined experience level and derived experience category
total = len(df)
matches = (df["experience_level"] == df["experience_category"]).sum()
mismatches = (total-matches)

match_pct = (matches / total) * 100
mismatch_pct = (mismatches / total) * 100

print(f"Match: {match_pct:.2f}%")
print(f"Mismatch: {mismatch_pct:.2f}%")

Match: 33.10%
Mismatch: 66.90%


#### Insight
Job-defined experience levels show weak alignment with experience categories derived from years of experience. Across all levels, the Senior category consistently dominates (~45%–46%; Entry: 1,607, Mid: 1,545, Senior: 1,571), while Entry and Mid categories remain closely distributed (~25%–28%). Overall, only 33% of records are correctly aligned, while the majority (67%) are mismatched. This indicates that experience levels are not consistently standardized and do not reliably reflect years of experience.

## Q4: Salary Premium by Technical Skills

### Average Salary by Skill Presence 

In [ ]:
# Computes salary difference between jobs with and without each skill
# and calculates the salary premium for each technical skill.
skill_results = []

for skill in skill_cols:
    avg_salary= df.groupby(skill)["salary"].mean()

    skill_results.append({
        "Skill" : skill,
        "No skill salary" : avg_salary[0],
        "Skill salary" : avg_salary[1],
        "Salary premium": avg_salary[1] - avg_salary[0]
    })
pd.DataFrame(skill_results).round(2)

,Skill,No skill salary,Skill salary,Salary premium
0,skills_python,113143.02,113741.71,598.68
1,skills_sql,113547.91,113329.87,-218.04
2,skills_ml,106020.81,120625.52,14604.71
3,skills_deep_learning,105856.69,121080.10,15223.40
4,skills_cloud,108501.30,118154.01,9652.71


#### Insight
In the job market, advanced technological skills show the highest salary premiums. The largest uplift (~15.2K) comes from deep learning, followed by machine learning (~14.6K) and cloud skills (~9.7K). In contrast, SQL shows no premium, while Python provides only a marginal gain (~600), indicating stronger demand for specialized AI and cloud skills over foundational tools.

## Q5: Skill Intensity vs Salary

### Salary Progression and Job Distribution by Total Skills

In [ ]:
# Analyze salary progression and job frequency across total required technical skills
df.groupby("total_skills")["salary"].agg(["mean", "count"]).round(2)

,mean,count
total_skills,,
0,91174.14,290
1,102254.46,1662
2,109404.32,3186
3,117000.66,3201
4,125038.05,1665
5,134490.97,341


#### Insight
Salary shows an overall upward trend with increasing skill count. Average salary rises from ~91K to ~134K as skill count increases from 0 to 5, representing a significant increase of approximately ~43K.
Each additional skill contributes roughly 7K–10K in salary growth, making the trend nearly linear. However, jobs requiring higher skill combinations (4–5 skills) are less common than roles requiring moderate skill sets (2–3 skills), indicating that highly technical roles are better paid but less frequently available in the job market.


##  Q6: Industry Demand vs Salary

In [ ]:
# Analyze mean salary and hiring demand across industries
df.groupby("company_industry").agg(
    avg_salary = ("salary", "mean"),
    hiring_demand = ("job_id", "count")
).round(0).sort_values("avg_salary", ascending=True)

,avg_salary,hiring_demand
company_industry,,
Healthcare,112127.0,1715
Finance,112376.0,1724
E-commerce,113494.0,1744
Retail,114081.0,1671
Education,114218.0,1704
Technology,114323.0,1787


#### Insight
Industries show very minimal variation in salary levels, ranging from ~112K to ~114K. Hiring demand is also relatively balanced across industries.. Technology slightly outperforms other sectors by offering the highest average salary (~114.3k) along with the highest hiring demand (1787 job postings). Overall, the results suggest that industry type has a limited influence on both compensation and hiring volume in this dataset.

## Q7: Job Opportunities vs Salary Across Countries

In [ ]:
# Analyze mean salary and job opportunites across countries
df.groupby("country").agg(
     avg_salary = ("salary", "mean"),
    job_opportunities = ("job_id", "count")
    ).round(0).sort_values("avg_salary", ascending=True)

,avg_salary,job_opportunities
country,,
Germany,111938.0,1498
India,112447.0,1470
UK,112722.0,1485
USA,113281.0,1459
Australia,114367.0,1455
Singapore,114541.0,1490
Canada,114783.0,1488


#### Insight
The impact of country on salary is relatively small compared to other factors such as technical skills and experience level. Average salaries range from ~111.9K in Germany to ~114.8K in Canada, representing a gap of approximately 2.8k. Canada, Singapore and Australia offer the highest average salaries among all countries. Job opportunities are also evenly distributed, with only a 43 postings difference across countries. Overall, country shows a moderate effect on salary but a limited influence on job availability in this dataset.

## Q8: Salary vs Education (Experience is Controlled)

In [ ]:
# Compute mean salary across education level by controlling experience using pivot table to evaluate whether education independently impacts salary beyond experience
df.pivot_table(
    values="salary",
    index="education_level",
    columns="experience_level",
    aggfunc="mean"
).round(0)

experience_level,Entry,Mid,Senior
education_level,,,
Bachelor,88904.0,112757.0,138015.0
Master,89293.0,114440.0,139129.0
PhD,89080.0,113538.0,137698.0


#### Insight
When experience level is controlled, education level show very limited influence on salary levels. The variation across education categories is minimal, generally within 1-2k. Master’s level consistently shows the highest average salary across all experience levels, slightly outperforming both Bachelor’s and PHD holders. In contrast, salary shows a clear and strong upward trend across experience levels, increasing significantly from Entry (~89K) to Mid (~113K) to Senior (~138–139K). This indicates that experience level is a primary driver of salary differences, while education has only a marginal effect and does not determine seniority and compensation levels.

## Q9: Hiring Urgency vs Salary

In [ ]:
# Analyze salary levels and hiring demand across differenct hiring urgency levels
df.groupby("hiring_urgency").agg(
    avg_salary = ("salary", "mean"),
    hiring_demand= ("job_id", "count")
).sort_values("avg_salary", ascending=True).round(0)

,avg_salary,hiring_demand
hiring_urgency,,
Medium,107830.0,2394
Low,108715.0,2756
High,118528.0,5195


#### Insight
Hiring urgency shows a moderate relationship with compensation. High urgency roles offer significantly higher average salaries (~118.5K) compared to medium (~107.8K) and low urgency (~108.7K), representing a salary premium of approximately 10K. Additionally, high urgency positions account for the majority of job postings, indicating that urgent hiring not only increases demand but is also associated with higher compensation in this dataset.


## Q10: Hiring Urgency vs Work Mode

### 10A: Count Analysis

In [ ]:
# Analyze hiring urgency distribution across remote, hybrid, and onsite jobs
pd.crosstab(
    df["remote_type"],
    df["hiring_urgency"]
)

hiring_urgency,High,Low,Medium
remote_type,,,
Hybrid,1711,924,785
Onsite,1722,893,797
Remote,1762,939,812


### 10B: Percentage Analysis

In [9]:
# Analyze percentage distribution of hiring urgency within each work arrangement
pd.crosstab(
    df["remote_type"],
    df["hiring_urgency"],
    normalize="index"
).multiply(100).round(2)

hiring_urgency,High,Low,Medium
remote_type,,,
Hybrid,50.03,27.02,22.95
Onsite,50.47,26.17,23.36
Remote,50.16,26.73,23.11


#### Insight
Hiring urgency shows an almost identical distribution across remote, hybrid, and onsite roles. Approximately 50% of jobs across all work arrangements fall under high urgency, while low and medium urgency remain consistently distributed at ~23% and ~26–27%, respectively. The minimal variation (generally within 1%) across work modes ,suggest that work arrangement has virtually no influence on hiring urgency in this dataset.

## AI Job Market Analysis — Executive Summary

### Objective
This analysis examines the key determinants of salary variation and hiring patterns in the AI job market, focusing on experience, technical skills, education, industry, geography, hiring urgency, and work arrangement.

The objective is to identify the primary drivers of compensation and hiring demand.

---

### Key Findings

#### Primary salary drivers
- Technical skills (strongest impact on compensation, with Deep learning, Machine Learning, and Cloud showing the highest premium)
- Experience level (consistent and dominant salary progression across all levels)
- Hiring urgency (strong association with higher compensation and increased job volume)

#### Secondary influence
- Geography (moderate variation in salary across countries)

#### Weak or negligible influence
- Industry classification (minimal salary differentiation)
- Education level (limited impact when experience levels are controlled)
- Work arrangement (no meaningful variation in salary or hiring urgency)

---

### Executive Conclusion
Salary trends in the AI job market are primarily driven by **technical specialization and experience levels**. Advanced skill sets and higher job expereince level consistently result in greater compensation, while traditional factors such as education, industry classification, and work arrangement show limited influence.

Overall, the findings indicate a **skill-driven and experience-driven compensation structure**, rather than one primarily determined by degrees or geographic factors.